# Stage 08 — Read-only Analytics Serving

**Goal:** לדעת לחשב ממוצע נכון ולהגן על מסד הנתונים משינויים דרך API.

**Setup:** Python 3.11+, Jupyter/VS Code; local synthetic examples only.

**CS bridge:** Explain the invariant and its failure mode before running code.

**Source:** https://www.sqlite.org/pragma.html#pragma_query_only

**Next:** Complete the graded lab under ../labs/ and pass an unseen transfer challenge.

## 1. Predict first

לפני Run All: כתוב מה אתה מצפה שה־assert יוכיח ומה אינו בודק. לאחר מכן הרץ את התא.

In [ ]:
import sqlite3
db=sqlite3.connect(":memory:")
db.execute("create table events(id text primary key,magnitude real)")
db.executemany("insert into events values (?,?)",[("A",2.0),("B",None),("C",4.0)])
db.commit()
total,measured,average=db.execute("select count(*),count(magnitude),avg(magnitude) from events").fetchone()
assert total==3 and measured==2 and average==3.0
print("Events:",total,"known magnitudes:",measured,"average:",average)

## 2. Failure mode / independent check

שנה פרמטר אחד בענף האישי והסבר מדוע הקוד נכשל או משנה תוצאה. החזר את הערך המקורי.

In [ ]:
db.execute("pragma query_only=on")
assert db.execute("select count(*) from events").fetchone()[0]==3
try:
    db.execute("update events set magnitude=10")
    raise AssertionError("write operation must fail")
except sqlite3.OperationalError:
    print("Read-only boundary succeeded; an HTTP service also needs allowlisting and auth design.")
db.close()

## 3. Transfer (no AI)

הצע נתיב API שמציג לפי אזור עם LIMIT ופרמטרים מאומתים. אילו נתונים אסור לחשוף?

אל תסמן את השלב כהושלם על בסיס מחברת זו בלבד; חובה לבצע את המעבדה ובדיקת המנטור.